In [1]:
#after obtaining API location of possible distribution in one country we should filter the results in the following steps
import geopandas as gpd
import pandas as pd
import ast
import osmnx as ox


In [2]:
#FILTER 1
#keep only the points that fall within Nigerian territory and saves the result to a new GeoPackage file.

points_file = "lpg_points_maps_nigeria_3857.gpkg"
boundary_file = "Country_boundaries.geojson"

points = gpd.read_file(points_file)
boundary = gpd.read_file(boundary_file)

if points.crs != boundary.crs:
    points = points.to_crs(boundary.crs)

nigeria = boundary.geometry.union_all()
points_nigeria = points[points.geometry.within(nigeria)]

print(f"FILTER 1 – Inside Nigeria")
print(f"  Original points: {len(points)}")
print(f"  Points inside Nigeria: {len(points_nigeria)}")
print(f"  Removed: {len(points)-len(points_nigeria)} ({100*(len(points)-len(points_nigeria))/len(points):.1f}%)\n")

FILTER 1 – Inside Nigeria
  Original points: 3812
  Points inside Nigeria: 3742
  Removed: 70 (1.8%)



In [3]:
#FILTER 2
#Filter out LNG-related places 


points_no_lng = points_nigeria[~points_nigeria['name'].str.contains('LNG', case=False, na=False)]

print(f"FILTER 2 – Exclude 'LNG' in name")
print(f"  Before: {len(points_nigeria)}")
print(f"  After: {len(points_no_lng)}")
print(f"  Removed: {len(points_nigeria)-len(points_no_lng)} ({100*(len(points_nigeria)-len(points_no_lng))/len(points_nigeria):.1f}%)\n")


FILTER 2 – Exclude 'LNG' in name
  Before: 3742
  After: 3736
  Removed: 6 (0.2%)



In [4]:
#FILTER 3
#Filter removing likely false positives, discard point with suspicius category with also NO key words in the name and NO gas_station as complementary type
            

suspect = [
    'store', 'car_repair', 'food', 'home_goods_store', 'furniture_store',
    'health', 'parking', 'restaurant', 'general_contractor',
    'grocery_or_supermarket', 'electronics_store', 'storage', 'finance',
    'pharmacy', 'supermarket', 'bakery', 'shopping_mall'
]

lpg_key_words = [
    "LPG refill station",
    "gas station with LPG",
    "LPG filling point",
    "LPG dealer",
    "gas cylinder exchange",
    "cooking gas",
    "LPG"
]

def parse_types(types_field):
    if isinstance(types_field, list):
        return types_field
    if isinstance(types_field, str):
        try:
            return ast.literal_eval(types_field)
        except (ValueError, SyntaxError):
            return [t.strip().strip("[]'\"") for t in types_field.split(',')]
    return []

def has_suspect(types_list):
    return any(cat in types_list for cat in suspect)

def has_gas_station(types_list):
    return 'gas_station' in types_list

def has_lpg_keyword(name):
    if pd.isna(name):
        return False
    name_lower = name.lower()
    return any(keyword.lower() in name_lower for keyword in lpg_key_words)

# Prepare types list
points_no_lng = points_no_lng.copy()
points_no_lng['types_list'] = points_no_lng['types'].apply(parse_types)

# Keep if not (suspect AND no gas_station AND no LPG keyword)
keep_mask = ~(
    points_no_lng['types_list'].apply(has_suspect) &
    ~points_no_lng['types_list'].apply(has_gas_station) &
    ~points_no_lng['name'].apply(has_lpg_keyword)
)

points_final = points_no_lng[keep_mask].copy()
points_removed_step3 = points_no_lng[~keep_mask].copy()

# Drop temporary column
points_final.drop(columns=['types_list'], inplace=True)
points_removed_step3.drop(columns=['types_list'], inplace=True)

print(f"FILTER 3 – Remove false positives")
print(f"  Before: {len(points_no_lng)}")
print(f"  After: {len(points_final)}")
print(f"  Removed: {len(points_no_lng)-len(points_final)} ({100*(len(points_no_lng)-len(points_final))/len(points_no_lng):.1f}%)\n")



FILTER 3 – Remove false positives
  Before: 3736
  After: 3523
  Removed: 213 (5.7%)



In [5]:
# FILTER 4
# Keep points within 150m of roads OR within 150m of a building

# load roads and buildings
roads = gpd.read_file('nigeria_roads_merged.gpkg') 
buildings = gpd.read_file('buildings_near_lpg_points_150m.gpkg')   # in EPSG:4326

# convert both to EPSG:3857 (metric)
roads_3857 = roads.to_crs('EPSG:3857')
buildings_3857 = buildings.to_crs('EPSG:3857')

# points are already in 3857 from previous filters and conversion
points_proj = points_final.copy()
points_proj = points_proj.to_crs('EPSG:3857')

# --- 1. points near roads (≤150m) ---
points_near_road = gpd.sjoin_nearest(
    points_proj,
    roads_3857[['geometry']],
    how='inner',
    max_distance=150,
    distance_col='dist_to_road'
)
points_near_road = points_near_road[~points_near_road.index.duplicated(keep='first')]
points_near_road = points_near_road.drop(columns=['dist_to_road'])

# --- 2. points far from roads (>150m) ---
far_idx = points_proj.index.difference(points_near_road.index)
points_far = points_proj.loc[far_idx]

# --- 3. among far points, keep those near buildings (≤150m) ---
points_far_near_buildings = gpd.sjoin_nearest(
    points_far,
    buildings_3857[['geometry']],
    how='inner',
    max_distance=150,
    distance_col='dist_to_building'
)
points_far_near_buildings = points_far_near_buildings[~points_far_near_buildings.index.duplicated(keep='first')]
points_far_near_buildings = points_far_near_buildings.drop(columns=['dist_to_building'])

# --- 4. final set: near road OR (far but near building) ---
points_final_filtered = pd.concat([points_near_road, points_far_near_buildings])

# Statistics
before = len(points_proj)
far_near_building = len(points_far_near_buildings)
after = len(points_final_filtered)
removed = before - after

print("FILTER 4 – Proximity to roads OR buildings")
print(f"  Points before filter: {before}")
print(f"  Points >150m from roads: {len(points_far)}")
print(f"  Points >150m from roads but within 150m of building: {far_near_building}")
print(f"  Points removed (isolated): {removed} ({100*removed/before:.1f}%)\n")

points_final = points_final_filtered


FILTER 4 – Proximity to roads OR buildings
  Points before filter: 3523
  Points >150m from roads: 57
  Points >150m from roads but within 150m of building: 24
  Points removed (isolated): 33 (0.9%)



In [6]:
# FILTER 5 – Remove places with no reviews
# Keep only points that have at least one review (user_ratings_total > 0)

before = len(points_final)
points_with_reviews = points_final[points_final['user_ratings_total'] > 0].copy()
after = len(points_with_reviews)
removed = before - after

print("FILTER 5 – Remove places with no reviews")
print(f"  Before: {before}")
print(f"  After: {after}")
print(f"  Removed: {removed} ({100*removed/before:.1f}%)\n")

# Update points_final to the filtered set
points_final = points_with_reviews

FILTER 5 – Remove places with no reviews
  Before: 3490
  After: 2833
  Removed: 657 (18.8%)



In [7]:
#save results

output_final = "lpg_points_nigeria_final_filtred.gpkg"
points_final.to_file(output_final, layer='lpg_points', driver='GPKG')
print(f"Final kept points saved to: {output_final}")



Final kept points saved to: lpg_points_nigeria_final_filtred.gpkg
